# 🏦 Exercícios SQL — CVM Data Lake
### Versão do Aluno | Big Data for Finance — FAE
**Banco:** PostgreSQL (DBeaver local)

**Tabelas utilizadas:**
- `layer_02_silver` — Tabelas que estão na camada Silver

**CNPJs de referência rápida:**
| Empresa | CNPJ |
|---|---|
| Petrobras | `33.000.167/0001-01` |
| Ambev | `07.526.557/0001-00` |
| Vale | `33.592.510/0001-54` |

---

## ⚙️ Configuração da Conexão

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv

load_dotenv()
print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


In [2]:
def create_db_engine():
    user = quote_plus(os.getenv('DB_USER', 'postgres'))
    password = quote_plus(os.getenv('DB_PASS', 'password'))
    host = os.getenv('DB_HOST', 'localhost')
    port = os.getenv('DB_PORT', '5432')
    dbname = os.getenv('DB_NAME', 'data_lake')
    connection_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(connection_str)

engine = create_db_engine()
print("🔌 Conexão com o banco de dados estabelecida!")

🔌 Conexão com o banco de dados estabelecida!


---
## Questão 1 — Primeira olhada na tabela Silver 🟢 Fácil
**Contexto:** A primeira coisa que um analista faz ao chegar em um novo banco é explorar o que existe dentro da tabela. A camada Silver já tem os dados do Balanço Patrimonial limpos, normalizados e prontos para análise.

**Pergunta:** Mostre as **10 primeiras linhas** da tabela `n1_dfp_cia_aberta_bp`, exibindo apenas as colunas: `CNPJ_CIA`, `DENOM_CIA`, `DT_REFER`, `CD_CONTA`, `DS_CONTA`, `VL_CONTA_TRATADO` e `IS_LEAF`.

In [ ]:
with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT "CNPJ_CIA", "DENOM_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA", "VL_CONTA_TRATADO", "IS_LEAF"
             FROM layer_02_silver.n1_dfp_cia_aberta_bp
        """),
        con=conn
    )

print(f"✅ Total de registros na tabela n1_dfp_cia_aberta_bp: {len(df_all):,}")
df_all.head()

📥 Carregando o universo completo de empresas da CVM...
✅ Total de registros na tabela n1_dfp_cia_aberta_bp: 180,461


,CNPJ_CIA,DENOM_CIA,DT_REFER,CD_CONTA,DS_CONTA,VL_CONTA_TRATADO,IS_LEAF
0,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2013-12-31,2.03.02,RESERVAS DE CAPITAL,11969000.0,False
1,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2013-12-31,2.03.02.06,ADIANTAMENTO PARA FUTURO AUMENTO DE CAPITAL,11969000.0,True
2,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2013-12-31,2.03.05,LUCROS/PREJUÍZOS ACUMULADOS,-94179000.0,True
3,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2013-12-31,2.03.06,AJUSTES DE AVALIAÇÃO PATRIMONIAL,48498000.0,False
4,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,2013-12-31,2.03.09,PARTICIPAÇÃO DOS ACIONISTAS NÃO CONTROLADORES,234420000.0,True


---
## Questão 2 — Volume e cobertura da base 🟢 Fácil
**Contexto:** Antes de qualquer análise, o analista precisa entender o tamanho e a cobertura da base: quantas linhas existem, quantas empresas distintas estão cobertas e qual o intervalo de datas disponível.

**Pergunta:** Em uma única query, calcule para a tabela `n1_dfp_cia_aberta_bp`:
- Total de linhas → `total_linhas`
- Quantidade de empresas distintas (por `CNPJ_CIA`) → `empresas_distintas`
- Quantidade de datas de referência distintas (`DT_REFER`) → `datas_distintas`

In [ ]:
with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT
                COUNT(*) AS TOTAL_LINHAS,
                COUNT(DISTINCT "CNPJ_CIA") AS QTDE_CNPJ,
                COUNT(DISTINCT "DT_REFER") AS QTDE_DT
            FROM layer_02_silver.n1_dfp_cia_aberta_bp
        """),
        con=conn
    )

df_all.head()


,total_linhas,qtde_cnpj,qtde_dt
0,180461,185,54


---
## Questão 3 — Distribuição de empresas por setor 🟢 Fácil
**Contexto:** A CVM classifica cada empresa listada em um setor de atividade (`SETOR_ATIV`). Entender essa distribuição é o ponto de partida para qualquer análise setorial comparativa. Use a tabela `n0_empresas_selecionadas`, que contém o cadastro completo das empresas do nosso universo.

**Pergunta:** Quantas empresas existem em cada `SETOR_ATIV`? Ordene do setor com **mais** empresas para o com **menos**.

In [9]:
with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT "SETOR_ATIV", COUNT(*) AS "SETOR_QTDE"
            FROM layer_02_silver.n1_dfp_cia_aberta_bp
            GROUP BY 1
            ORDER BY 2 DESC
        """),
        con=conn
    )

df_all.head()

,SETOR_ATIV,SETOR_QTDE
0,"Construção Civil, Mat. Constr. e Decoração",31425
1,Comércio (Atacado e Varejo),18762
2,Energia Elétrica,18205
3,"Máquinas, Equipamentos, Veículos e Peças",14645
4,Têxtil e Vestuário,13850


---
## Questão 4 — Petrobras: Ativo e Passivo mais recentes 🟢 Fácil
**Contexto:** No Balanço Patrimonial da CVM, `CD_CONTA = '1'` é o **Ativo Total** e `CD_CONTA = '2'` é o **Passivo + Patrimônio Líquido Total**. A equação fundamental do BP é: **Ativo = Passivo + PL** — qualquer diferença indica erro de dados.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`), mostre o valor (em bilhões de reais) das contas `'1'` e `'2'` para a **data de referência mais recente** disponível. Use uma subconsulta escalar com `MAX("DT_REFER")` para encontrar a data mais recente.

*Dica: `ROUND((valor / 1e9)::NUMERIC, 2)` converte para bilhões com 2 casas.*

In [13]:
with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT "CD_CONTA", ROUND((SUM("VL_CONTA_TRATADO") / 1e9)::NUMERIC, 2) AS "VL_SALDO"
                FROM layer_02_silver.n1_dfp_cia_aberta_bp
                WHERE "CNPJ_CIA" = '33.000.167/0001-01'
                AND "CD_CONTA" in ('1','2')
                AND "DT_REFER" = (
                        SELECT MAX("DT_REFER")
                        FROM layer_02_silver.n1_dfp_cia_aberta_bp
                        WHERE "CNPJ_CIA" = '33.000.167/0001-01'
                        AND "CD_CONTA" in ('1','2'))
                GROUP BY 1
        """),
        con=conn
    )

df_all.head()

,CD_CONTA,VL_SALDO
0,1,1124.8
1,2,1124.8


---
## Questão 5 — IS_LEAF: folhas vs. agregadores 🟢 Fácil
**Contexto:** `IS_LEAF = TRUE` marca contas **analíticas** (sem filhos) — são as únicas que devem ser somadas em qualquer ferramenta de BI, pois evitam dupla contagem. `IS_LEAF = FALSE` são contas **sintéticas** (pais) que já incorporam os valores de todas as suas contas filhas. Somar tudo geraria dupla contagem.

**Pergunta:** Para a **Petrobras** (`CNPJ_CIA = '33.000.167/0001-01'`) em `DT_REFER = '2023-12-31'`, quantas contas são `IS_LEAF = TRUE` e quantas são `IS_LEAF = FALSE`? Use `GROUP BY "IS_LEAF"`.

In [14]:
with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT "IS_LEAF", COUNT(*) AS "QUANTIDADE"
            FROM layer_02_silver.n1_dfp_cia_aberta_bp
            WHERE "CNPJ_CIA" = '33.000.167/0001-01'
            AND "DT_REFER" = '2023-12-31'
            GROUP BY 1
        """),
        con=conn
    )

df_all.head()

,IS_LEAF,QUANTIDADE
0,False,51
1,True,19


---
## Questão 6 — Top 10 maiores empresas por Ativo Total em 2023 🟡 Médio
**Contexto:** Rankings de tamanho de balanço são análises clássicas de mercado de capitais. O Ativo Total (`CD_CONTA = '1'`) resume o tamanho total de uma empresa — tudo que ela possui e tem direito a receber.

**Pergunta:** Liste as **10 empresas com maior Ativo Total** em `DT_REFER = '2023-12-31'`. Exiba `DENOM_CIA`, `SETOR_ATIV` e o valor em **bilhões** (2 casas decimais). Ordene do maior para o menor.

*Dica: `ROUND((expr)::NUMERIC, 2)` — o cast para `NUMERIC` é obrigatório no PostgreSQL quando a coluna é `DOUBLE PRECISION`.*

In [15]:
with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT "DENOM_CIA", "SETOR_ATIV", 
                ROUND(("VL_CONTA_TRATADO" / 1000000000)::NUMERIC, 2) AS "ATIVO_TOTAL_BILHOES"
            FROM layer_02_silver.n1_dfp_cia_aberta_bp
            WHERE "DT_REFER" = '2023-12-31' 
            AND "CD_CONTA" = '1'
            ORDER BY "VL_CONTA_TRATADO" DESC
            LIMIT 10
        """),
        con=conn
    )

df_all




,DENOM_CIA,SETOR_ATIV,ATIVO_TOTAL_BILHOES
0,PETROLEO BRASILEIRO S.A. PETROBRAS,Petróleo e Gás,1050.89
1,VALE S.A.,Extração Mineral,455.98
2,JBS S.A.,Alimentos,206.13
3,SUZANO S.A.,Papel e Celulose,143.59
4,COSAN S.A.,"Agricultura (Açúcar, Álcool e Cana)",139.87
5,AMBEV S.A.,Bebidas e Fumo,132.64
6,MARFRIG GLOBAL FOODS S.A.,Alimentos,130.95
7,TELEFÔNICA BRASIL S.A,Telecomunicações,120.74
8,EQUATORIAL S.A.,Energia Elétrica,103.64
9,BRASKEM S.A.,Petroquímicos e Borracha,91.74


---
## Questão 7 — Evolução anual do Ativo Total da Ambev 🟡 Médio
**Contexto:** Acompanhar a evolução do balanço ao longo dos anos revela a trajetória de crescimento (ou retração) de uma empresa. `DT_REFER` é do tipo `TIMESTAMP` — use `EXTRACT(YEAR FROM ...)` para extrair o ano.

**Pergunta:** Para a **Ambev** (`CNPJ_CIA = '07.526.557/0001-00'`), mostre a evolução anual do **Ativo Total** (`CD_CONTA = '1'`). Extraia o ano de `DT_REFER`, exiba o valor em bilhões e ordene do ano mais antigo ao mais recente.

In [16]:
with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT EXTRACT(YEAR FROM "DT_REFER") AS "ANO",
                ROUND(("VL_CONTA_TRATADO" / 1000000000.0)::NUMERIC, 2) AS "VALOR_TOTAL_BILHOES"
            FROM layer_02_silver.n1_dfp_cia_aberta_bp
            WHERE "CNPJ_CIA" = '07.526.557/0001-00' 
            AND "CD_CONTA" = '1'
            ORDER BY "ANO" ASC;
        """),
        con=conn
    )

df_all

,ANO,VALOR_TOTAL_BILHOES
0,2013.0,68.67
1,2014.0,72.14
2,2015.0,90.18
3,2016.0,83.84
4,2017.0,86.85
5,2018.0,94.13
6,2019.0,101.74
7,2020.0,125.20
8,2021.0,138.60
9,2022.0,137.96


---
## Questão 8 — Contas reconstruídas pelo pipeline 🟡 Médio
**Contexto:** `FLAG_RECONSTRUCAO = TRUE` indica que uma linha foi criada **sinteticamente** pelo pipeline — a empresa não reportou aquela conta e o pipeline reconstruiu o valor somando os filhos. Mais contas reconstruídas = mais "buracos" no dado original enviado à CVM.

**Pergunta:** Em **toda a base histórica**, quais são as **10 combinações de empresa e data** com mais contas reconstruídas no BP? Mostre `DT_REFER`, `DENOM_CIA` e o total de contas de cada empresa. Use `SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END)` para contar. Agrupe por `"DT_REFER"` e `"DENOM_CIA"`. Ordene da mais para a menos reconstruída.


In [17]:

with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT "DT_REFER", "DENOM_CIA", 
                SUM(CASE WHEN "FLAG_RECONSTRUCAO" = TRUE THEN 1 ELSE 0 END) AS "QTD_RECONSTRUIDAS"
            FROM layer_02_silver.n1_dfp_cia_aberta_bp
            GROUP BY "DT_REFER", "DENOM_CIA"
            ORDER BY "QTD_RECONSTRUIDAS" DESC
            LIMIT 10
        """),
        con=conn
    )

df_all

,DT_REFER,DENOM_CIA,QTD_RECONSTRUIDAS
0,2022-12-31,CESP - CIA ENERGETICA DE SAO PAULO,15
1,2021-12-31,VIVER INCORPORADORA E CONSTRUTORA S.A.,2
2,2017-12-31,CIA SIDERURGICA NACIONAL,1
3,2010-12-31,USINAS SID DE MINAS GERAIS S.A.-USIMINAS,1
4,2016-12-31,CIA SIDERURGICA NACIONAL,1
5,2021-12-31,INTERNATIONAL MEAL COMPANY ALIMENTACAO S.A.,1
6,2022-12-31,GRUPO CASAS BAHIA S.A.,1
7,2022-12-31,CIA BRASILEIRA DE DISTRIBUICAO,1
8,2021-12-31,CVC BRASIL OPERADORA E AGÊNCIA DE VIAGENS S.A.,1
9,2018-12-31,PRIO S.A.,0


---
## Questão 9 — Base analítica de Receita Bruta por empresa via JOIN 🟡 Médio
**Contexto:** Antes de agregar, o analista precisa da base analítica: uma linha por empresa com seu setor e receita. Com esse DataFrame em mãos, o Pandas consegue gerar estatísticas descritivas por grupo com `.describe()`.

**Pergunta:** Faça um `JOIN` entre `n1_dfp_cia_aberta_dre` e `n0_empresas_selecionadas`. Para `DT_REFER = '2023-12-31'` e `CD_CONTA = '3.01'` (Receita Bruta), traga uma linha por empresa com `CNPJ_CIA`, `DENOM_CIA`, `SETOR_ATIV` e o valor em bilhões. Depois use `.groupby` + `.describe()` para ver as estatísticas por setor.


In [18]:
with engine.connect() as conn:
    df_all = pd.read_sql(
        text("""
            SELECT d."CNPJ_CIA", d."DENOM_CIA", e."SETOR_ATIV", 
                ROUND((d."VL_CONTA_TRATADO" / 1e9)::NUMERIC, 2) AS receita_bilhoes
            FROM layer_02_silver.n1_dfp_cia_aberta_dre d
                JOIN layer_02_silver.n0_empresas_selecionadas e 
                    ON d."CNPJ_CIA" = e."CNPJ_CIA"
            WHERE d."CD_CONTA" = '3.01' 
            AND d."DT_REFER" = '2023-12-31';
        """),
        con=conn
    )

df_all

,CNPJ_CIA,DENOM_CIA,SETOR_ATIV,receita_bilhoes
0,00.070.698/0001-11,COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB,Energia Elétrica,0.35
1,01.107.327/0001-20,BBM LOGÍSTICA S.A.,Serviços Transporte e Logística,1.61
2,01.545.826/0001-07,GAFISA S.A.,"Construção Civil, Mat. Constr. e Decoração",1.10
3,02.302.101/0001-42,EMAE - EMPRESA METROP.AGUAS ENERGIA S.A.,Energia Elétrica,0.60
4,02.351.144/0001-18,TEGMA GESTAO LOGISTICA S.A.,Serviços Transporte e Logística,1.58
...,...,...,...,...
166,92.754.738/0001-62,LOJAS RENNER S.A.,Comércio (Atacado e Varejo),13.65
167,92.781.335/0001-02,TAURUS ARMAS S.A.,"Máquinas, Equipamentos, Veículos e Peças",1.78
168,96.418.264/0218-02,LOJAS QUERO QUERO S.A.,Comércio (Atacado e Varejo),2.40
169,97.191.902/0001-94,CONSERVAS ODERICH S.A.,Alimentos,0.78
